# medcalc-kit tutorial

This notebook walks through the whole library on MedCalc-Bench-Verified:

1. [Data](#1.-Data): load the benchmark, look at one example, make splits
2. [Calculators](#2.-Calculators): the reference computation, step by step
3. [Probes](#3.-Probes): what each probe (E, U, K, S, A, E2E) asks and how it scores an answer
4. [Building items](#4.-Building-items): every probe for many examples at once
5. [Running a model](#5.-Running-a-model): a real model (Qwen2.5-3B-Instruct) on a subset
6. [Scoring and analysis](#6.-Scoring-and-analysis): artifacts, competence, errors, the evaluation report
7. [Further analyses](#7.-Further-analyses): a smaller token budget, interventions, the LLM judge
8. [Customising](#8.-Customising): templates, budgets, variants, scoring options
9. [Low-level tools](#9.-Low-level-tools): safe expressions, units, answer parsing

Everything returns pandas DataFrames. To run it: `conda env create -f environment.yml`, then pick the kernel
"Python (medcalc-kit)". Without a GPU, section 5 falls back to a simulated model and every other cell runs the same.
The API reference is in `docs/api.md` and the concepts in `docs/guide.md`.

In [1]:
import medcalc_kit as mk
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)
print("medcalc-kit", mk.__version__)

# Settings of section 5
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # any chat model of the Hugging Face Hub
EXAMPLES_PER_CALCULATOR = 2              # size of the subset the model answers (33 calculators)
BATCH_SIZE = 16
RUN_DIR = "outputs/tutorial"             # items, generations and report of this notebook
OUTPUT = f"{RUN_DIR}/generations.jsonl"  # generations are appended here (delete it to regenerate)

medcalc-kit 0.5.0


## 1. Data

`mk.load()` downloads MedCalc-Bench-Verified from the Hugging Face Hub at a pinned revision (`mk.data.DEFAULT_REVISION`),
then reads it from the cache: 55 calculators. The end-to-end task covers all of them; the step probes need the
registry's computation, so they cover the 33 equation-based calculators (`equation_only()`).

In [2]:
full = mk.load()
ds = full.equation_only()
print(full)
ds

/home/merveil/miniconda3/envs/medcalc-kit/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MedCalcDataset(11638 examples, 55 calculators; train: 10538, test: 1100)


MedCalcDataset(8532 examples, 33 calculators; train: 7872, test: 660)

In [3]:
ds.calculators()          # one row per calculator, number of examples per split

,Calculator ID,Calculator Name,Category,test,train
0,2,Creatinine Clearance (Cockcroft-Gault Equation),lab test,20,131
1,3,CKD-EPI Equations for Glomerular Filtration Rate,lab test,20,483
2,5,Mean Arterial Pressure (MAP),physical,20,927
3,6,Body Mass Index (BMI),physical,20,496
4,7,Calcium Correction for Hypoalbuminemia,lab test,20,185
5,9,MDRD GFR Equation,lab test,20,275
6,10,Ideal Body Weight,physical,20,599
7,11,QTc Bazett Calculator,physical,20,80
8,19,Fibrosis-4 (FIB-4) Index for Liver Fibrosis,lab test,20,90
9,22,Maintenance Fluids Calculations,physical,20,842


A `MedCalcDataset` filters by calculator, category and split, and gives `Example`s by position or by id.
`ds.df` is the underlying DataFrame with the dataset's original columns.

In [4]:
test = ds.filter(split="test")
ex = test.filter(calculator=2)[0]        # creatinine clearance (Cockcroft-Gault)
ex

Example('test:2:1', calculator 2 'Creatinine Clearance (Cockcroft-Gault Equation)', ground truth 25.24)

In [5]:
print(ex.note[:600], "...\n")
print("Question:", ex.question[:200], "...\n")
print("Annotated inputs:", ex.entities)
print("Ground truth:", ex.ground_truth, "accepted interval:", (round(ex.lower, 3), round(ex.upper, 3)))

An 87-year-old man was admitted to our hospital for anorexia for several days, high-grade fever from the previous day, and liver dysfunction. Of note, he had a history of hypertension, diabetes mellitus (DM), and angina. Physical examination findings included: clear consciousness; height, 163 cm; weight, 48 kg; blood pressure, 66/40 mmHg; heart rate, 75/min; respiratory rate, 22/min; oxygen saturation of peripheral artery, 96%; and body temperature, 38.1 °C. He had no surface lymphadenopathy. Laboratory findings included: white blood cell (WBC) count, 4.2 × 109/L; hemoglobin, 9.6 g/dL; platele ...

Question: What is the patient's Creatinine Clearance using the Cockroft-Gault Equation in terms of mL/min? You should use the patient's adjusted body weight in kg instead of the patient's actual body weight if  ...

Annotated inputs: {'sex': 'Male', 'age': [87, 'years'], 'weight': [48.0, 'kg'], 'height': [163.0, 'cm'], 'creatinine': [1.4, 'mg/dL']}
Ground truth: 25.2381 accepted interval: (2

### Splits

The `split` column is free: besides "train" and "test", make any split. `split_off` moves part of a split to a new
one, stratified by calculator; with `exclude_leaks=True` it never takes an example whose note also appears in the
test set. Every method returns a new dataset.

In [6]:
custom = ds.split_off("dev", fraction=0.1, exclude_leaks=True, seed=0)
custom.splits()

,examples,calculators
split,,
dev,751,33
test,660,33
train,7121,33


In [7]:
custom.filter(split=["dev", "test"], category="lab test", where=lambda df: df["Note Type"] == "Extracted").splits()

,examples,calculators
split,,
dev,263,15
test,326,18


k-fold cross-validation, hand-picked splits, and saving a split to restore it later on a fresh copy of the benchmark:

In [8]:
folds = ds.folds(5)                                            # the i-th dataset has fold i of train in "dev"
print([len(f.filter(split="dev")) for f in folds])

hard = custom.assign_split(custom.filter(split="test", calculator=[23, 24]).example_ids, "hard")
print(hard)

path = custom.save_splits("outputs/splits/dev10.csv")
print(mk.load().equation_only().with_splits(path).splits().equals(custom.splits()))

[1585, 1580, 1573, 1570, 1564]
MedCalcDataset(8532 examples, 33 calculators; train: 7121, dev: 751, test: 620, hard: 40)


True


## 2. Calculators

`mk.derive(calculator_id, entities)` converts the inputs to canonical units, resolves the branch that applies to
this patient (here: which weight to use, depending on the BMI) and computes each step.

In [9]:
d = mk.derive(ex.calculator_id, ex.entities)     # same as ex.derive()
print("branches:", d.branches)
print("answer:", d.answer)
d.to_frame()

branches: ['ibw:male', 'bmi<18.5:actual']
answer: 25.238095238095237


,kind,name,expr,value
0,input,age,,87.000000
1,input,weight,,48.000000
2,input,height,,163.000000
3,input,scr,,1.400000
4,coefficient,k_sex,,1.000000
5,step,bmi,weight / (height / 100)^2,18.066167
6,step,height_in,height / 2.54,64.173228
7,step,ibw,50 + 2.3 * (height_in - 60),59.598425
8,step,dosing_weight,weight,48.000000
9,step,crcl,((140 - age) * dosing_weight * k_sex) / (72 * scr),25.238095


The registry covers 33 calculators (`mk.calculators.table()`); `docs/calculators.md` lists the entities each one
expects. Errors say what is missing:

In [10]:
mk.calculators.table().head(8)

,calculator_id,name,reference_deviation
0,2,Creatinine Clearance (Cockcroft-Gault Equation),
1,3,CKD-EPI Equations for Glomerular Filtration Rate,
2,5,Mean Arterial Pressure (MAP),
3,6,Body Mass Index (BMI),
4,7,Calcium Correction for Hypoalbuminemia,
5,9,MDRD GFR Equation,
6,10,Ideal Body Weight,
7,11,QTc Bazett Calculator,


In [11]:
try:
    mk.derive(6, {"weight": [154, "lbs"]})
except KeyError as err:
    print(err)
mk.derive(6, {"weight": [154, "lbs"], "height": [5, "ft", 9, "in"]}).to_frame()

"calculator 6 (Body Mass Index (BMI)) needs the entity 'height'; got ['weight']"


,kind,name,expr,value
0,input,weight,,69.853168
1,input,height,,175.260000
2,step,bmi,weight / (height / 100)^2,22.741573


The registry reproduces every test answer of the benchmark. `verify` checks it on any set of examples:

In [12]:
per_example, by_calculator = mk.calculators.verify_registry(test)
print(f"{per_example.within_bounds.mean():.1%} of {len(per_example)} test answers reproduced")
by_calculator.head()

100.0% of 660 test answers reproduced


,calculator_id,calculator_name,split,n,within_bounds,n_fail,n_exceptions,median_rel_error,max_rel_error
0,10,Ideal Body Weight,test,20,1.0,0,0,1.357379e-06,0.000002
1,11,QTc Bazett Calculator,test,20,1.0,0,0,1.160406e-06,0.000005
2,19,Fibrosis-4 (FIB-4) Index for Liver Fibrosis,test,20,1.0,0,0,7.044382e-07,0.000003
3,2,Creatinine Clearance (Cockcroft-Gault Equation),test,20,1.0,0,0,6.067747e-07,0.000002
4,22,Maintenance Fluids Calculations,test,20,1.0,0,0,0.000000e+00,0.000000


## 3. Probes

Each probe module has `build(example)` (the prompt and the reference) and `score(item, text)` (the verdict).
`mk.score` dispatches to the right probe, `mk.probes.prompt(item)` shows the chat, and `mk.reference_answer(item)`
gives an answer the scorer accepts, i.e. the expected format.

In [13]:
pd.DataFrame([(p.name, p.title, p.measures) for p in mk.probes.PROBES.values()],
             columns=["probe", "name", "measures"])

,probe,name,measures
0,E,Extraction,finding the input values (with units) in the patient note
1,U,Unit conversion,converting a patient value to the unit the formula uses
2,K,Knowledge,"writing the formula (general, or for this patient) without computing"
3,S,Substitution,"plugging the values into a given formula, without computing"
4,A,Arithmetic,computing a fully written numerical expression
5,E2E,End to end,the original benchmark task


### E: extraction

The model reports each input with its unit as written in the note. Values are converted to canonical units before
comparison, so `154 lbs` and `70 kg` are the same weight.

In [14]:
e = mk.probes.extraction.build(ex)
print(mk.probes.prompt(e)[-900:])

(Fig. d), but negative for CD3 (Fig. e), CD10, bcl-6, and cyclin D1, with a Ki-67 labeling index (Ki-67 LI) of 16% (Fig. f). The causes of death were thus determined as (1) DIC and (2) ALF associated with diffuse large B-cell lymphoma.

Task: extract the following information about the patient from the note. Use the values at the patient's first admission, before any treatment. Report each numeric value exactly as written in the note together with its unit (do not convert units). Use null if a value is not mentioned.

Fields:
- "sex": "Male" or "Female"
- "age": {"value": <number>, "unit": <unit as written>}
- "weight": {"value": <number>, "unit": <unit as written>}
- "height": {"value": <number>, "unit": <unit as written>}
- "creatinine": {"value": <number>, "unit": <unit as written>}

You may think briefly, then end your answer with a single JSON object containing exactly these fields.


In [15]:
print("expected format:", mk.reference_answer(e))
answer = '{"sex": "Male", "age": {"value": 87, "unit": "years"}, "weight": {"value": 105.8, "unit": "lbs"}, ' \
         '"height": {"value": 163, "unit": "cm"}, "creatinine": {"value": 1.4, "unit": "µmol/L"}}'
verdict = mk.score(e, answer)
print({k: verdict[k] for k in ("correct", "field_accuracy", "error_type")})
pd.Series(verdict["fields"], name="field result")

expected format: {"sex": "Male", "age": {"value": 87, "unit": "years"}, "weight": {"value": 48.0, "unit": "kg"}, "height": {"value": 163.0, "unit": "cm"}, "creatinine": {"value": 1.4, "unit": "mg/dL"}}
{'correct': False, 'field_accuracy': 0.8, 'error_type': 'unit_error'}


sex              correct
age              correct
weight           correct
height           correct
creatinine    unit_error
Name: field result, dtype: str

### U: unit conversion

One of the patient's values is given in another plausible unit (drawn reproducibly from `conversion.UNIT_TABLE`)
and must be converted to the unit of the registry's formula. The prompt names the substance but never its molar mass.

In [16]:
u = mk.probes.conversion.build(ex, field="creatinine")
print(mk.probes.prompt(u).split("[user] ")[1])
ref = u["reference"]
candidates = {"converted": ref["gold_value"], "copied back": ref["source_value"],
              "factor the wrong way": ref["source_value"] ** 2 / ref["gold_value"], "other": 3.0}
pd.DataFrame([{"answer": round(v, 4), **mk.score(u, f'{{"value": {v}}}')} for v in candidates.values()],
             index=list(candidates))

A patient's serum creatinine is 123.8 µmol/L.

Convert this value to mg/dL. Work carefully without any tool, then end your answer with a JSON object of the form {"value": <number in mg/dL with at least 4 significant digits>}

,answer,format_valid,correct,error_type,rel_error
converted,1.4004,True,True,NaN,0.000000
copied back,123.8000,True,False,unconverted,87.401697
factor the wrong way,10944.1301,True,False,inverse_factor,7813.860088
other,3.0000,True,False,wrong_value,1.142206


U exists for the examples with a convertible value: 24 of the 33 equation-based calculators.

In [17]:
pd.DataFrame(mk.probes.conversion.UNIT_TABLE, index=["substance", "analyte", "formula's unit", "other units"]).T

,substance,analyte,formula's unit,other units
creatinine,serum creatinine,creatinine,mg/dL,"(µmol/L,)"
Urine creatinine,urine creatinine,creatinine,mg/dL,"(µmol/L, mmol/L)"
Sodium,serum sodium,sodium,mEq/L,"(mmol/L,)"
Urine sodium,urine sodium,sodium,mEq/L,"(mmol/L,)"
Chloride,serum chloride,chloride,mEq/L,"(mmol/L,)"
Bicarbonate,serum bicarbonate,bicarbonate,mEq/L,"(mmol/L,)"
Glucose,serum glucose,glucose,mg/dL,"(mmol/L,)"
Calcium,serum calcium,calcium,mg/dL,"(mmol/L, mEq/L)"
Blood Urea Nitrogen (BUN),blood urea nitrogen,bun,mg/dL,"(mmol/L,)"
Albumin,serum albumin,albumin,g/dL,"(g/L,)"


### K: knowledge of the formula

The model writes the formula with the variable names, without the patient's values. It is evaluated at the
patient's point and at up to 50 other patients, and must match the registry everywhere. Here, two candidate
answers: the textbook Cockcroft-Gault (actual weight), and the benchmark's rule (the weight depends on the BMI).

In [18]:
pool = mk.data.usable_train_pool(ds)                         # leakage-free train patients, used as test points
k = mk.probes.knowledge.build(ex, pool[pool["Calculator ID"] == "2"]["entities"].tolist())
print(mk.probes.prompt(k)[-1200:])

adjusted body weight in kg instead of the patient's actual body weight if the patient is overweight or obese based on their BMI. If the patient's BMI's normal, set their adjusted body weight to the minimum of the ideal body and actual weight. If the patient is underweight, please set their adjusted body weight to their actual body weight.

You are NOT given the patient's measurements. Write the general formula needed to answer this task, using only the variables listed below. Their units are fixed as stated; include any unit conversion inside your formula.

Known patient characteristics:
- sex: Male

Variables:
- age: patient age (years)
- weight: body weight (kg)
- height: height (cm)
- scr: serum creatinine (mg/dL)

If the formula depends on thresholds or categories, give one case per branch, each with a condition written with these variables (for example "x <= 0.9"), and use "otherwise" for the remaining case. Use only numbers, the variables, + - * / ^, parentheses and the functions

In [19]:
textbook = '{"cases": [{"condition": "always", "formula": "(140 - age) * weight / (72 * scr)"}]}'
ibw = "50 + 2.3 * (height / 2.54 - 60)"
bmi = "weight / (height / 100)^2"
rule = ('{"cases": ['
        f'{{"condition": "{bmi} < 18.5", "formula": "(140 - age) * weight / (72 * scr)"}}, '
        f'{{"condition": "18.5 <= {bmi} <= 24.9", "formula": "(140 - age) * min(weight, {ibw}) / (72 * scr)"}}, '
        f'{{"condition": "{bmi} > 24.9", "formula": "(140 - age) * ({ibw} + 0.4 * (weight - ({ibw}))) / (72 * scr)"}}]}}')
for name, text in [("textbook", textbook), ("benchmark rule", rule)]:
    v = mk.score(k, text)
    print(f"{name:15} correct={v['correct']}  error={v['error_type']}  points passed={v['point_pass_rate']:.0%}")

textbook        correct=False  error=wrong_elsewhere  points passed=47%
benchmark rule  correct=True  error=None  points passed=100%


`details=True` shows each test point: the textbook formula is right for this (underweight) patient, wrong for
heavier ones.

In [20]:
points = pd.DataFrame(mk.score(k, textbook, details=True)["points"])
points.round(2).head(8)

,age,weight,height,scr,registry,model,ok
0,87.0,48.0,163.0,1.40,25.24,25.24,True
1,16.0,55.0,162.8,0.57,166.18,166.18,True
2,56.0,68.0,176.0,1.00,79.33,79.33,True
3,34.0,74.0,174.0,3.10,33.03,35.14,False
4,67.0,62.0,157.0,0.56,103.74,112.25,False
5,61.0,46.0,154.0,1.30,38.82,38.82,True
6,51.0,79.0,165.1,1.35,62.72,72.34,False
7,43.0,115.0,198.0,7.82,17.35,19.80,False


By default the formula must be right at every point. `min_pass_rate` lowers the share required; a formula wrong at the
patient's own point is never accepted.

In [21]:
for threshold in (1.0, 0.9, 0.4):
    v = mk.score(k, textbook, min_pass_rate=threshold)
    print(f"min_pass_rate={threshold}: correct={v['correct']} (points passed {v['point_pass_rate']:.0%})")

min_pass_rate=1.0: correct=False (points passed 47%)
min_pass_rate=0.9: correct=False (points passed 47%)
min_pass_rate=0.4: correct=True (points passed 47%)


### S: substitution

The model gets the formula of the patient's branch and the values, and must substitute them without computing.
Reordering is fine; computing part of it is not.

In [22]:
s = mk.probes.substitution.build(ex)
print(mk.probes.prompt(s).split("[user] ")[1])
gold = s["reference"]["gold_expression"]
candidates = {
    "faithful copy": gold,
    "reordered": "((140 - 87) * 48 * 1) / (1.4 * 72)",
    "partly computed": "53 * 48 * 1 / (72 * 1.4)",
    "variable left": "((140 - age) * 48 * 1) / (72 * 1.4)",
}
pd.DataFrame([{"answer": a, **{k: v for k, v in mk.score(s, f'{{"expression": "{a}"}}').items()
                              if k in ("correct", "error_type", "value_matches")}}
              for a in candidates.values()], index=list(candidates))

Formula for Creatinine Clearance (Cockcroft-Gault Equation):
crcl = (140 - age) * weight * 1 / (72 * scr)

Variable values:
- age = 87   (patient age (years))
- weight = 48   (body weight (kg))
- scr = 1.4   (serum creatinine (mg/dL))

Substitute the values into the formula. Do NOT compute or simplify anything: copy the formula exactly, replacing each variable by its value.

End your answer with a JSON object of the form {"expression": "<formula with the values substituted>"}


,answer,correct,error_type,value_matches
faithful copy,(140 - 87) * 48 * 1 / (72 * 1.4),True,NaN,True
reordered,((140 - 87) * 48 * 1) / (1.4 * 72),True,NaN,True
partly computed,53 * 48 * 1 / (72 * 1.4),False,premature_computation,True
variable left,((140 - age) * 48 * 1) / (72 * 1.4),False,unsubstituted_variable,False


Variant S+U: some values are given in another unit, and the model converts them while substituting.

In [23]:
su = mk.probes.substitution.build(ex, value_units={"scr": "µmol/L"})
print(su["probe"], "|", [ln for ln in mk.probes.prompt(su).splitlines() if ln.startswith("- ")])
raw = su["reference"]["gold_expression"].replace(f"{su['reference']['values']['scr']:g}", "123.8")
print(mk.score(su, mk.reference_answer(su))["correct"], mk.score(su, f'{{"expression": "{raw}"}}')["error_type"])

S+U | ['- age = 87   (patient age (years))', '- weight = 48   (body weight (kg))', '- scr = 123.8 µmol/L   (serum creatinine (mg/dL))']
True unconverted_value


### A: arithmetic

The fully substituted expression, to compute. Correct within 1 % (`correct`) and 0.1 % (`correct_strict`).

In [24]:
a = mk.probes.arithmetic.build(ex)
print(a["reference"]["expression"], "=", a["reference"]["gold_value"])
pd.DataFrame([mk.score(a, f'{{"result": {x}}}') for x in (25.24, 25.4, 30)], index=[25.24, 25.4, 30])

(140 - 87) * 48 * 1 / (72 * 1.4) = 25.238095238095237


,format_valid,correct,correct_strict,error_type,rel_error
25.24,True,True,True,NaN,0.000075
25.40,True,True,False,NaN,0.006415
30.00,True,False,False,arithmetic_error,0.188679


### E2E: the original task

The note and the question; the answer must fall within the benchmark's interval. E2E covers all 55 calculators,
including dates.

In [25]:
e2e = mk.probes.end_to_end.build(ex)
pd.DataFrame([mk.score(e2e, t) for t in ('{"answer": 25.2}', '{"answer": 27}', "about 25 mL/min")],
             index=["25.2", "27", "no JSON"])

,format_valid,correct,error_type,rel_error
25.2,True,True,NaN,0.001510
27,True,False,out_of_bounds,0.069811
no JSON,False,False,format_invalid,NaN


In [26]:
due = full.filter(split="test", category="date")[0]
item = mk.probes.end_to_end.build(due)
print(due.calculator_name, "| ground truth:", due.ground_truth)
print(mk.probes.prompt(item).splitlines()[-1])
[mk.score(item, t)["error_type"] for t in (mk.reference_answer(item), '{"answer": "2000-01-01"}')]

Estimated Due Date | ground truth: 09/11/2014
Think step by step, then end your answer with a JSON object of the form {"answer": "<MM/DD/YYYY>"}


[None, 'wrong_date']

Interventions: `provide` adds the correct values and/or formula of the patient's branch to the question, to measure
what each step costs end to end (section 7).

In [27]:
vf = mk.probes.end_to_end.build(ex, provide=("values", "formula"))
print(vf["probe"])
print(mk.probes.prompt(vf).split("Question: ")[1][-700:])

E2E+VF
MI's normal, set their adjusted body weight to the minimum of the ideal body and actual weight. If the patient is underweight, please set their adjusted body weight to their actual body weight. You should use the patient's medical values and health status when they were first admitted to the hospital prior to any treatment.

Correct values of the variables, in the units of the formula:
- age = 87 (patient age (years))
- weight = 48 (body weight (kg))
- height = 163 (height (cm))
- scr = 1.4 (serum creatinine (mg/dL))

Formula that applies to this patient:
crcl = (140 - age) * weight * 1 / (72 * scr)

Think step by step, then end your answer with a JSON object of the form {"answer": <number>}


## 4. Building items

`ProbeSuite` builds every probe for many examples and returns one row per item. `schema=ds` makes E ask the same
fields for every example of a calculator; `k_pool` gives K real patients as test points. Here K runs in `patient`
mode (the model sees the values and writes the formula that applies to this patient).

In [28]:
subset = test.sample(EXAMPLES_PER_CALCULATOR, per_calculator=True, seed=0)
suite = mk.ProbeSuite({"E": {}, "U": {}, "K": {"mode": "patient"}, "S": {}, "A": {}, "E2E": {}}, schema=ds,
                      k_pool=pool)
items = suite.build(subset)
print(len(subset), "examples ->", len(items), "items")
items[["item_id", "probe", "calculator_name", "max_new_tokens"]].head()

66 examples -> 378 items


,item_id,probe,calculator_name,max_new_tokens
0,test:2:19:E,E,Creatinine Clearance (Cockcroft-Gault Equation),768
1,test:2:2:E,E,Creatinine Clearance (Cockcroft-Gault Equation),768
2,test:3:39:E,E,CKD-EPI Equations for Glomerular Filtration Rate,768
3,test:3:22:E,E,CKD-EPI Equations for Glomerular Filtration Rate,768
4,test:5:79:E,E,Mean Arterial Pressure (MAP),768


In [29]:
items.groupby("probe", sort=False).agg(items=("item_id", "size"), token_budget=("max_new_tokens", "first"),
                                       prompt_chars=("messages", lambda m: m.map(lambda x: len(x[-1]["content"])).mean()))

,items,token_budget,prompt_chars
probe,,,
E,66,768,3376.409091
U,48,768,216.666667
K,66,1024,803.454545
S,66,768,473.151515
A,66,1024,342.045455
E2E,66,2048,3101.227273


Items can be saved and reloaded (one JSONL file per probe), e.g. to generate on another machine:

In [30]:
counts = mk.probes.write_items(items, f"{RUN_DIR}/items")
print(counts)
mk.probes.read_items(f"{RUN_DIR}/items").probe.value_counts()

{'E': 66, 'U': 48, 'K': 66, 'S': 66, 'A': 66, 'E2E': 66}


probe
A         66
E         66
E2E+F     66
E2E+V     66
E2E+VF    66
E2E       66
K         66
S         66
U         48
Name: count, dtype: int64

Before spending GPU time, check the pipeline with the reference answers: every item must be scored correct.

In [31]:
oracle = pd.DataFrame({"item_id": items.item_id, "text": [mk.reference_answer(it) for _, it in items.iterrows()]})
mk.summary(mk.score_records(oracle, items))

,n_units,well_formed,competence,overall
probe,,,,
A,66.0,1.0,1.0,1.0
E,66.0,1.0,1.0,1.0
E2E,66.0,1.0,1.0,1.0
K,66.0,1.0,1.0,1.0
S,66.0,1.0,1.0,1.0
U,48.0,1.0,1.0,1.0


## 5. Running a model

`run_items(items, path, model)` generates an answer for every item and appends one JSON line per item to `path`. It
generates each distinct prompt once, batches prompts of the same budget, and skips items already in the file, so an
interrupted run resumes where it stopped.

`model` can be a backend with `generate_batch` (here `HFBackend`, greedy decoding with the chat template), any
object with that method, or a plain function `generate(messages, max_new_tokens)`. If no GPU is available, a
simulated model answers instead: the reference answer, degraded at random per probe.

In [32]:
import random


def pick_device():
    try:
        import torch
    except ImportError:
        return None
    if not torch.cuda.is_available():
        return None
    free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
    best = max(range(len(free)), key=free.__getitem__)
    return f"cuda:{best}" if free[best] > 12e9 else None


class SimulatedModel:
    """Stand-in when no GPU is available: reference answers, with errors and artifacts at random."""
    model_id = "simulated"
    error_rate = {"E": 0.4, "U": 0.4, "K": 0.5, "S": 0.3, "A": 0.2, "E2E": 0.6}

    def __init__(self, items, seed=0):
        self.by_prompt = {mk.generation.prompt_key(it["messages"], it["max_new_tokens"]): it
                          for _, it in items.iterrows()}
        self.rng = random.Random(seed)

    def generate(self, messages, max_new_tokens):
        item = self.by_prompt.get(mk.generation.prompt_key(messages, max_new_tokens))
        if item is None:
            return "I cannot answer this.", 100, 5
        text = "Let me work through this. " + mk.reference_answer(item)
        r = self.rng.random()
        if r < 0.05:
            return "", 0, 0
        if r < self.error_rate.get(item.get("task", item["probe"]), 0.5):
            text = text.replace("1", "7", 1) if "1" in text else "I am not sure."
        return text, 100, len(text.split())


device = pick_device()
if device:
    import transformers
    from medcalc_kit.generation.hf import HFBackend

    transformers.utils.logging.disable_progress_bar()
    model = HFBackend(MODEL_ID, device=device)
else:
    model = SimulatedModel(items)
print("model:", model.model_id, "on", device or "cpu (simulated)")

model: Qwen/Qwen2.5-3B-Instruct on cuda:0


In [33]:
import time

t0 = time.time()
n_new = mk.run_items(items, OUTPUT, model, batch_size=BATCH_SIZE)
print(f"{n_new} new records in {time.time() - t0:.0f} s")
records = mk.read_records(OUTPUT)
records[["item_id", "model_id", "n_new_tokens", "truncated", "seconds", "text"]].head()

0 new records in 0 s


,item_id,model_id,n_new_tokens,truncated,seconds,text
0,test:10:162:S,Qwen/Qwen2.5-3B-Instruct,28,False,0.071,"{""expression"": ""50 + 2.3 * (177 / 2.54 - 60)""}"
1,test:10:179:S,Qwen/Qwen2.5-3B-Instruct,30,False,0.071,"{""expression"": ""45.5 + 2.3 * (165 / 2.54 - 60)""}"
2,test:22:322:S,Qwen/Qwen2.5-3B-Instruct,16,False,0.071,"{""expression"": ""60加上(64减去20)""}"
3,test:22:339:S,Qwen/Qwen2.5-3B-Instruct,18,False,0.071,"{""expression"": ""60加上(39.5减去20)""}"
4,test:24:1002:S,Qwen/Qwen2.5-3B-Instruct,22,False,0.071,"{""expression"": ""162.935 * 1 / 26.67""}"


In [34]:
print(mk.probes.prompt(items.iloc[0])[-300:])
print("\n--- model answer ---\n")
print(records.set_index("item_id").loc[items.iloc[0].item_id, "text"])

<unit as written>}
- "height": {"value": <number>, "unit": <unit as written>}
- "creatinine": {"value": <number>, "unit": <unit as written>}
- "age": {"value": <number>, "unit": <unit as written>}

You may think briefly, then end your answer with a single JSON object containing exactly these fields.

--- model answer ---

{
  "sex": "Female",
  "weight": {"value": 61.0, "unit": "kg"},
  "height": {"value": 155.5, "unit": "cm"},
  "creatinine": {"value": 1.4, "unit": "mg/dl"}
}


## 6. Scoring and analysis

`score_records` scores every answer: its artifact category (`empty`, `truncated`, `degenerate`, `format_invalid`,
`well_formed`), whether it is correct (only well-formed answers can be), and its error type. `condition` adds
constant columns, used later to compare models; `details=True` keeps the text and the full verdict.

In [35]:
scores = mk.score_records(OUTPUT, items, condition={"model": model.model_id}, details=True)
scores[["item_id", "probe", "category", "correct", "error_type", "n_new_tokens"]].head(10)

,item_id,probe,category,correct,error_type,n_new_tokens
0,test:10:162:S,S,well_formed,True,NaN,28
1,test:10:179:S,S,well_formed,True,NaN,30
2,test:22:322:S,S,format_invalid,False,unparsable_expression,16
3,test:22:339:S,S,format_invalid,False,unparsable_expression,18
4,test:24:1002:S,S,well_formed,True,NaN,22
5,test:24:1019:S,S,well_formed,True,NaN,25
6,test:6:99:S,S,well_formed,True,NaN,19
7,test:6:82:S,S,well_formed,True,NaN,23
8,test:11:199:S,S,format_invalid,False,format_invalid,19
9,test:11:182:S,S,format_invalid,False,format_invalid,19


`summary` gives, per probe: the share of well-formed answers, the accuracy among them (**competence**), and the
overall accuracy (`overall = well_formed × competence`).

In [36]:
mk.summary(scores).style.format({"n_units": "{:.0f}", "well_formed": "{:.0%}", "competence": "{:.0%}", "overall": "{:.0%}"})

,n_units,well_formed,competence,overall
probe,,,,
A,66,80%,81%,65%
E,66,100%,85%,85%
E2E,66,100%,18%,18%
K,66,83%,5%,5%
S,66,80%,94%,76%
U,48,100%,75%,75%


In [37]:
pd.crosstab(scores.probe, scores.category)                 # where the artifacts are

category,format_invalid,truncated,well_formed
probe,,,
A,11,2,53
E,0,0,66
E2E,0,0,66
K,10,1,55
S,13,0,53
U,0,0,48


In [38]:
(scores[scores.well_formed & ~scores.correct]
 .groupby(["probe", "error_type"]).size().rename("answers").reset_index())   # which errors among usable answers

,probe,error_type,answers
0,A,arithmetic_error,10
1,E,missing,1
2,E,unit_error,5
3,E,wrong_value,4
4,E2E,out_of_bounds,54
5,K,wrong_at_patient_point,52
6,S,accidental_value_match,1
7,S,structure_or_assignment_error,1
8,S,too_ambiguous,1
9,U,wrong_value,12


Per calculator: which calculators fail at which step. A calculator can be right end to end only if every step is
right, so this table shows where the chain breaks.

In [39]:
by_calc = scores.pivot_table(index="calculator_id", columns="probe", values="correct", aggfunc="mean")
by_calc = by_calc[[p for p in ["E", "U", "K", "S", "A", "E2E"] if p in by_calc]]
names = mk.calculators.table().set_index("calculator_id").name
by_calc.index = [f"{i} {names[i][:35]}" for i in by_calc.index]
by_calc.style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1)

probe,E,U,K,S,A,E2E
10 Ideal Body Weight,50%,100%,0%,100%,50%,50%
11 QTc Bazett Calculator,100%,nan%,0%,0%,100%,0%
19 Fibrosis-4 (FIB-4) Index for Liver,50%,nan%,0%,100%,100%,0%
2 Creatinine Clearance (Cockcroft-Gau,0%,100%,0%,0%,50%,0%
22 Maintenance Fluids Calculations,100%,100%,0%,0%,100%,0%
23 MELD Na (UNOS/OPTN),50%,0%,0%,50%,0%,0%
24 Steroid Conversion Calculator,100%,nan%,0%,100%,100%,0%
26 Sodium Correction for Hyperglycemia,100%,100%,0%,0%,0%,50%
3 CKD-EPI Equations for Glomerular Fi,100%,0%,0%,100%,0%,0%
30 Serum Osmolality,100%,100%,0%,100%,100%,0%


Drill into one failure: the answer and the verdict.

In [40]:
fails = scores[scores.well_formed & ~scores.correct]
if len(fails):
    row = fails.iloc[0]
    print(row.item_id, "->", row.error_type, "\n")
    print(row.text[-800:], "\n")
    print({k: v for k, v in row.verdict.items() if k != "points"})

test:23:342:S -> structure_or_assignment_error 

{"expression": "round(round(0.957 * ln(min(max(1.6, 1), 4)) + 0.378 * ln(max(1.11, 1)) + 1.12 * ln(max(0.9, 1)), 1) * 10)"} 

{'format_valid': True, 'correct': False, 'error_type': 'structure_or_assignment_error', 'value_matches': False, 'missing_variables': [], 'match': None}


### Comparing conditions

Score several runs with different `condition`s, concatenate, and group. Here the model is compared with the oracle
(reference answers), which is the ceiling.

In [41]:
both = pd.concat([scores, mk.score_records(oracle, items, condition={"model": "oracle"})])
mk.summary(both, by=("model", "probe")).unstack("model")["overall"].style.format("{:.0%}")

model,Qwen/Qwen2.5-3B-Instruct,oracle
probe,,
A,65%,100%
E,85%,100%
E2E,18%,100%
K,5%,100%
S,76%,100%
U,75%,100%


Paired comparison on the same items:

In [42]:
paired = both.pivot_table(index=["probe", "item_id"], columns="model", values="correct", aggfunc="first")
paired.groupby("probe").mean().style.format("{:.0%}")

model,Qwen/Qwen2.5-3B-Instruct,oracle
probe,,
A,65%,100%
E,85%,100%
E2E,18%,100%
K,5%,100%
S,76%,100%
U,75%,100%


### The evaluation report

`evaluate_model` chains all of this. Given the items built above (`items=`), it scores the answers already generated
in `RUN_DIR` (nothing is generated again) and returns the report: score of each step, first failing step of each
example, consistency of the decomposition.

In [43]:
report = mk.evaluate_model(model, RUN_DIR, items=items, dataset=full)
report

MedCalc-Bench evaluation of Qwen/Qwen2.5-3B-Instruct
  general (end to end): 18.2% of 66 examples (33 calculators)
  step by step (66 examples of equation-based calculators):
    E    extraction          84.8%
    U    unit conversion     75.0%
    K    formula knowledge    4.5%
    S    substitution        75.8%
    A    arithmetic          65.2%
    E2E  end to end          18.2%
  consistency: E2E correct for 67% of the examples whose steps all succeed, 16% when a step fails

In [44]:
report.first_failures.style.format({"share": "{:.0%}", "e2e_accuracy": "{:.0%}"})

,examples,share,e2e_accuracy
first_failure,,,
E,10,15%,0%
U,9,14%,11%
K,44,67%,20%
none,3,5%,67%


## 7. Further analyses

### A smaller token budget, without generating again

With greedy decoding, the first tokens of an answer do not depend on the budget: cutting the saved answers tells
what a smaller budget would have given.

In [45]:
class Words:                                         # stand-in tokenizer when no Hugging Face model is loaded
    def encode(self, text): return text.split()
    def decode(self, ids): return " ".join(ids)


tokenizer = model.tokenizer if device else Words()
records = mk.read_records(OUTPUT)
budgets = {"E2E": 256, "K": 256, "A": 256}
cut = mk.evaluate.truncate_records(records, items, budgets, tokenizer)
pd.DataFrame({"generated budget": mk.summary(mk.score_records(records, items))["overall"],
              "256 tokens": mk.summary(mk.score_records(cut, items))["overall"]}).style.format("{:.0%}")

,generated budget,256 tokens
probe,,
A,65%,44%
E,85%,85%
E2E,18%,6%
K,5%,5%
S,76%,76%
U,75%,75%


### Interventions on the end-to-end task

The same examples, with the correct values, the correct formula, or both given in the question. The gap with plain E2E
measures what each step costs the model end to end.

In [46]:
interventions = mk.ProbeSuite({"E2E": {}, "E2E+V": {"task": "E2E", "provide": ("values",)},
                               "E2E+F": {"task": "E2E", "provide": ("formula",)},
                               "E2E+VF": {"task": "E2E", "provide": ("values", "formula")}}).build(subset)
if not device:
    model = SimulatedModel(pd.concat([items, interventions]))
t0 = time.time()
int_report = mk.evaluate_model(model, RUN_DIR, items=interventions, dataset=full, batch_size=BATCH_SIZE)
print(f"{time.time() - t0:.0f} s")
int_report.by_label.loc[["E2E", "E2E+V", "E2E+F", "E2E+VF"]].style.format(
    {"well_formed": "{:.0%}", "competence": "{:.0%}", "accuracy": "{:.0%}"})

0 s


,n,well_formed,competence,accuracy
label,,,,
E2E,66,100%,18%,18%
E2E+V,66,98%,22%,21%
E2E+F,66,98%,58%,58%
E2E+VF,66,100%,68%,68%


### LLM evaluation, next to the automatic one

An LLM judge can grade every answer too: it reads the task, the registry's reference answer and the probe's grading
rule (for E2E, only the final answer against the ground truth), and says correct or incorrect. Comparing it with the
automatic scoring measures how far it can be trusted. `evaluate_model(model, judge=...)` runs it on every answer (see
`evaluate_model.ipynb`); here, what the judge reads for one answer of this run:

In [47]:
records = mk.read_records(OUTPUT)
s_item = items[items.probe == "S"].iloc[0]
rec = records[records.item_id == s_item.item_id].iloc[0].to_dict()
print(mk.probes.prompt(mk.judge.build(s_item, rec)).split("[user] ")[1])
print("\nautomatic evaluation:", mk.score(s_item, rec["text"])["correct"])

You grade the answer of an assistant to a clinical calculation task, against the reference answer.

Task given to the assistant:
<<<
Formula for Creatinine Clearance (Cockcroft-Gault Equation):
crcl = (140 - age) * (45.5 + 2.3 * (height / 2.54 - 60) + 0.4 * (weight - (45.5 + 2.3 * (height / 2.54 - 60)))) * 0.85 / (72 * scr)

Variable values:
- age = 49   (patient age (years))
- weight = 61   (body weight (kg))
- height = 155.5   (height (cm))
- scr = 1.4   (serum creatinine (mg/dL))

Substitute the values into the formula. Do NOT compute or simplify anything: copy the formula exactly, replacing each variable by its value.

End your answer with a JSON object of the form {"expression": "<formula with the values substituted>"}
>>>

Reference answer: (140 - 49) * (45.5 + 2.3 * (155.5 / 2.54 - 60) + 0.4 * (61 - (45.5 + 2.3 * (155.5 / 2.54 - 60)))) * 0.85 / (72 * 1.4)

Grading rule: Correct if the answer is the given formula with each variable replaced by its value and nothing computed or si

### A model modified in memory

`HFBackend.from_model` wraps a model already loaded, for example pruned or quantized in this session, without
loading it again; `evaluate_model` then evaluates it like any other.

In [48]:
if device:
    from medcalc_kit.generation.hf import HFBackend
    wrapped = HFBackend.from_model(model.model, model.tokenizer, model_id="qwen2.5-3b (wrapped)")
    print(wrapped.model_id, wrapped.device, wrapped.dtype)
    print(wrapped.generate([{"role": "user", "content": "Reply with the number 4 only."}], 8))

qwen2.5-3b (wrapped) cuda:0 bfloat16


('4', 37, 1)


## 8. Customising

Every setting is an argument. Prompts are templates with `{placeholders}` (listed next to each module's
`DEFAULT_TEMPLATE`); other braces, such as JSON examples, are kept as written.

In [49]:
print(mk.probes.arithmetic.DEFAULT_TEMPLATE)
custom = mk.probes.arithmetic.build(ex, template='Calculate {expression}. Reply with {"result": <number>} only.',
                                    system_prompt=None, max_new_tokens=256, digits=3)
print("\n" + mk.probes.prompt(custom))

Compute the value of the following expression (ln is the natural logarithm, log10 the base-10 logarithm, round(x, n) rounds x to n decimals, ^ is exponentiation):

{expression}

Work carefully without any tool. End your answer with a JSON object of the form {"result": <number with at least 4 significant digits>}

[user] Calculate (140 - 87) * 48 * 1 / (72 * 1.4). Reply with {"result": <number>} only.


Several variants of a probe in one suite: the label becomes the probe name, `task` says which probe scores it.

In [50]:
variants = mk.ProbeSuite({"K_general": {"task": "K"}, "K_patient": {"task": "K", "mode": "patient"},
                          "A_short": {"task": "A", "max_new_tokens": 256}}, k_pool=pool)
v_items = variants.build(subset.filter(calculator=[2, 6]))
v_items[["item_id", "probe", "task", "max_new_tokens"]]

,item_id,probe,task,max_new_tokens
0,test:2:19:K_general,K_general,K,1024
1,test:2:2:K_general,K_general,K,1024
2,test:6:99:K_general,K_general,K,1024
3,test:6:82:K_general,K_general,K,1024
4,test:2:19:K_patient,K_patient,K,1024
5,test:2:2:K_patient,K_patient,K,1024
6,test:6:99:K_patient,K_patient,K,1024
7,test:6:82:K_patient,K_patient,K,1024
8,test:2:19:A_short,A_short,A,256
9,test:2:2:A_short,A_short,A,256


Scoring options apply per label or per task, e.g. a stricter arithmetic tolerance:

In [51]:
strict = mk.score_records(OUTPUT, items, scoring={"A": {"rel_tol": 0.001}}, condition={"model": model.model_id})
pd.DataFrame({"default (1 %)": mk.summary(scores)["overall"], "strict (0.1 %)": mk.summary(strict)["overall"]})

,default (1 %),strict (0.1 %)
probe,,
A,0.651515,0.409091
E,0.848485,0.848485
E2E,0.181818,0.181818
K,0.045455,0.045455
S,0.757576,0.757576
U,0.750000,0.750000


## 9. Low-level tools

`expr` is the safe expression engine used to score K and S: a whitelisted grammar, never `eval`.

In [52]:
from medcalc_kit import expr, parsing, units

print(expr.evaluate("(140 - age) * weight / (72 * scr)", {"age": 87, "weight": 48, "scr": 1.4}))
print(sorted(expr.variables("sqrt(weight * height / 3600)")))
print(expr.canonical("a - (b + c)") == expr.canonical("a - c - b"))
try:
    expr.evaluate("__import__('os').system('ls')")
except expr.ExprError as err:
    print("refused:", err)

25.238095238095237
['height', 'weight']
True
refused: unexpected character "'" at position 11


`units` converts the benchmark's annotations; `parsing` extracts the answer from free text.

In [53]:
print(units.concentration(1.4, "mg/dL", "creatinine", "µmol/L"), units.convert(70, "kg", "lbs"))
print(units.height_cm([5, "ft", 9, "in"]), units.weight_kg([154, "lbs"]))
print(parsing.extract_json('I computed it. ```json\n{"result": "25.2 mL/min"}\n```'))
print(parsing.to_number("25.2 mL/min"))

123.76237623762374 154.32370941286442
175.26 69.853168
{'result': '25.2 mL/min'}
25.2
